# Step 1: Raw Data — The Landing Zone

A data lake starts by accepting data **as-is**: whatever format the source system produces, with no upfront modeling. This is the opposite of a data warehouse, where data is transformed and validated *before* it is loaded (schema-on-write).

In this notebook we generate a synthetic sales export — `date, region, product, quantity, revenue` — as a single CSV file. CSV is deliberately our starting point: row-based, uncompressed, human-readable, and exactly what a POS or ERP export typically looks like when it lands in a lake's raw zone.

In [ ]:
import os
from pathlib import Path

# Make sure we run from the repository root, regardless of the
# notebook's working directory
while not (Path.cwd() / "requirements.txt").exists():
    os.chdir("..")
print("Working directory:", Path.cwd())

## Generate the raw CSV

`scripts/generate_data.py` writes 300,000 rows in a few chunks. Expect a file of roughly 10-15 MB — small enough to generate and query in seconds, and small enough that you could open it in a spreadsheet and page through it by hand, but still large enough to make format and query differences clearly measurable.

In [ ]:
!python scripts/generate_data.py --out lake/raw/sales.csv

## Look at the raw data

Open the file browser and find `lake/raw/sales.csv`. Note its size — we'll compare it against Parquet in the next notebook.

In [ ]:
import pandas as pd

csv_path = Path("lake/raw/sales.csv")
size_mb = csv_path.stat().st_size / (1024 * 1024)
print(f"{csv_path} is {size_mb:.1f} MB")

pd.read_csv(csv_path, nrows=10)

## Why CSV is a bad *permanent* home for this data

- **Row-based**: to read just the `revenue` column, every engine still has to scan every byte of every row.
- **No compression**: repeated values like region and product names are stored in full, every single time.
- **No embedded schema**: types (`quantity` is an int, `revenue` is a float) have to be *guessed* on every read.
- **No partitioning**: a query for a single month still means reading the entire file.

These four points are exactly what the next two notebooks fix — first with a better file format (Parquet), then with a query engine (DuckDB) that knows how to exploit it.